# Scanners

The same notebook as [`../scanners.ipynb`](../scanners.ipynb), written against
[`ib_async`](https://github.com/ib-api-reloaded/ib_async) instead of the TWS API
shape. The library is unmodified and installed as usual; `ibx.ib_async.attach`
replaces the one layer of it that expects a socket to a gateway.

What can be scanned for, and one scan run.

## Connecting

`IB.connect` was written for a gateway, so it takes a host, a port and a client
id. Here it takes none of them: the credentials go to `attach`, and there is no
local process to reach.

`ib.sleep()` rather than `time.sleep()` throughout. The library's loop runs on
this thread, and a plain sleep stops it — every stream then reads as dead.

In [ ]:
import os
from dotenv import load_dotenv
from ib_async import IB, util
import ibx.ib_async

util.startLoop()
load_dotenv()

ib = ibx.ib_async.attach(
    IB(),
    username=os.environ["IB_USERNAME"],
    password=os.environ["IB_PASSWORD"],
    paper=True,
)
ib.connect()          # names no host: there is no gateway to name

print(f"connected: {ib.isConnected()}")
print(f"accounts:  {ib.managedAccounts()}")

## What can be scanned

The venue states its own parameters, as one XML document. It is large; this
reads the scan codes out of it.

In [ ]:
import xml.etree.ElementTree as ET

xml = ib.reqScannerParameters()
print(f"{len(xml):,} bytes")

root = ET.fromstring(xml)
codes = [e.findtext("scanCode") for e in root.iter("ScanType")]
print(f"{len(codes)} scan codes, first twelve:\n")
for c in codes[:12]:
    print(" ", c)

## One scan

A scan names what to look at, and what to rank it by.

In [ ]:
from ib_async import ScannerSubscription

sub = ScannerSubscription(
    instrument="STK",
    locationCode="STK.US.MAJOR",
    scanCode="TOP_PERC_GAIN",
    abovePrice=5,
    aboveVolume=100_000,
)

rows = ib.reqScannerData(sub)
print(f"{len(rows)} results\n")
for r in rows[:15]:
    c = r.contractDetails.contract
    print(f"{r.rank:>3}  {c.symbol:8} {c.primaryExchange:10} {c.currency}")

## As a frame

In [ ]:
util.df([
    {
        "rank": r.rank,
        "symbol": r.contractDetails.contract.symbol,
        "exchange": r.contractDetails.contract.primaryExchange,
    }
    for r in rows
]).head(15)

In [ ]:
ib.disconnect()